<a href="https://colab.research.google.com/github/AgathaCRuiz/Voice-Powered-Labs/blob/main/02_SQL_Voice_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install groq gTTS pandas -q

import os
from google.colab import userdata

# Configurações Iniciais
GROQ_API_KEY = userdata.get('GROQ_API_KEY')
MODELO_GROQ = 'llama-3.1-8b-instant' # Versão atualizada e suportada

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 4.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:
import sqlite3
import pandas as pd

# Criar banco de dados na memória
conn = sqlite3.connect(':memory:', check_same_thread=False)
cursor = conn.cursor()

# Criar tabela de produtos
cursor.execute('''
    CREATE TABLE produtos (
        id INTEGER PRIMARY KEY,
        nome TEXT,
        categoria TEXT,
        preco REAL,
        estoque INTEGER
    )
''')

# Dados para teste
dados = [
    (1, 'Notebook Dell', 'Eletrônicos', 4500.0, 12),
    (2, 'iPhone 15', 'Eletrônicos', 7200.0, 5),
    (3, 'Teclado Mecânico', 'Periféricos', 350.0, 20),
    (4, 'Monitor 27 pol', 'Eletrônicos', 1800.0, 8),
    (5, 'Cadeira Gamer', 'Móveis', 1500.0, 10),
    (6, 'Mouse Sem Fio', 'Periféricos', 120.0, 45)
]
cursor.executemany('INSERT INTO produtos VALUES (?,?,?,?,?)', dados)
conn.commit()
print("✅ Banco de dados SQL pronto para consultas!")

✅ Banco de dados SQL pronto para consultas!


In [ ]:
import whisper
import json
from groq import Groq
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
from gtts import gTTS

# Carregar Whisper
whisper_model = whisper.load_model('base')
groq_client = Groq(api_key=GROQ_API_KEY)

# JavaScript para o microfone (Correção para não travar)
RECORD_JS = """
var record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await new Promise(r => setTimeout(r, time))
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    reader = new FileReader()
    reader.readAsDataURL(blob)
    reader.onloadend = () => resolve(reader.result)
  }
  recorder.stop()
})
"""

def falar(texto):
    tts = gTTS(text=texto, lang='pt-br')
    tts.save('res.mp3')
    display(Audio('res.mp3', autoplay=True))

def gravar_audio(segundos=10):
    display(Javascript(RECORD_JS))
    resultado = output.eval_js(f'record({segundos * 1000})')
    audio = b64decode(resultado.split(',')[1])
    with open('voce.wav', 'wb') as f: f.write(audio)
    return 'voce.wav'

# Prompt focado em SQL
SYSTEM_PROMPT = """
Você é um tradutor de voz para SQL. A tabela chama-se 'produtos' (colunas: id, nome, categoria, preco, estoque).
Retorne APENAS um JSON: {"sql": "sua query", "msg": "frase curta de confirmação"}.
Exemplo: {"sql": "SELECT * FROM produtos ORDER BY preco DESC LIMIT 10", "msg": "Buscando os 10 mais caros."}
"""

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 91.2MiB/s]


In [ ]:
import time

def assistente_dados_por_voz(segundos=10):
    print('\n🎙️  Aguardando seu comando SQL...')
    falar("Pode falar.")

    # Pausa para o "Pode falar" não ser gravado pelo microfone
    time.sleep(1.5)

    # 1. Gravar e Transcrever
    arquivo = gravar_audio(segundos)
    transcricao = whisper_model.transcribe(arquivo, language='pt')['text']
    print(f'📝 Você disse: "{transcricao}"')

    # 2. Entender com Groq
    res = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "system", "content": SYSTEM_PROMPT},
                  {"role": "user", "content": transcricao}],
        response_format={"type": "json_object"}
    )

    info = json.loads(res.choices[0].message.content)

    # --- CORREÇÃO AQUI ---
    print(f"🔊 IA Confirmando: {info['msg']}")
    falar(info['msg'])

    # Pausa de segurança para a IA terminar de falar a confirmação
    # antes de processar o SQL e falar o resultado
    time.sleep(2.5)

    print(f"🔍 SQL Gerado: {info['sql']}")

    # 3. Executar no Banco de Dados
    try:
        df = pd.read_sql_query(info['sql'], conn)

        if not df.empty:
            display(df)
            # Segunda fala apenas após o display
            mensagem_final = f"Encontrei {len(df)} resultados."
            print(f"🔊 IA: {mensagem_final}")
            falar(mensagem_final)
        else:
            msg_vazia = "Não encontrei nada com esse critério."
            print(f"🔊 IA: {msg_vazia}")
            falar(msg_vazia)

    except Exception as e:
        print(f"❌ Erro: {e}")
        falar("Erro ao executar consulta.")

# --- EXECUTAR ---
assistente_dados_por_voz()


🎙️  Aguardando seu comando SQL...


<IPython.core.display.Javascript object>

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 Você disse: " Me mostra todo o znacategoria eletrônicos."


🔊 IA Confirmando: Buscando produtos Eletrônicos.


🔍 SQL Gerado: SELECT * FROM produtos WHERE categoria = 'Eletrônicos'


,id,nome,categoria,preco,estoque
0,1,Notebook Dell,Eletrônicos,4500.0,12
1,2,iPhone 15,Eletrônicos,7200.0,5
2,4,Monitor 27 pol,Eletrônicos,1800.0,8


🔊 IA: Encontrei 3 resultados.
